<a href="https://colab.research.google.com/github/Padmashree02/Intrusion-detection-Computer-Vision/blob/main/Main_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#imported required libraries

import cv2
import numpy as np
from moviepy.editor import *

  if event.key is 'enter':



In [6]:
#Load the video

vid=cv2.VideoCapture('/content/CCTV_footage.mp4')

#Pre-processing step
frame_w=int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))   #results- width
frame_h=int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))  #results- height
fps=int(vid.get(cv2.CAP_PROP_FPS))               #results- frames per second

#Procedure to form the file for saving the output video
size=(frame_w,frame_h)
size_quad=(int(2*frame_w),int(2*frame_h))

#file name in which video to be saved
vid_out_file='Output_video_alert.mp4'
vid_out_quad_file='Output_video_quad.mp4'

vid_out=cv2.VideoWriter(vid_out_file,cv2.VideoWriter_fourcc(*'XVID'),fps,size)
vid_out_quad=cv2.VideoWriter(vid_out_quad_file,cv2.VideoWriter_fourcc(*'XVID'),fps,size_quad)


In [3]:
#Define the background subtractor with the history as 200 (by considering 200 previous frames)
bg_sub=cv2.createBackgroundSubtractorKNN(history=200)

In [4]:
#Necessary defination of values

ksize=(5,5)                     #For erosion purpose
max_contours=3
min_contour_area_thresh=0.01    #For detection purpose

#Color codes
yellow=(0,255,255)
red=(0,0,255)

#Frame size (width * height)
frame_area=frame_w*frame_h

In [7]:
#Object detection using Contour method

#Capturing the video into frames
while True:

  reteval,frame=vid.read()
  if frame is None:
    break

  #Form foreground mask
  fg_mask=bg_sub.apply(frame)

  #Performs errosion (to remove noise)
  fg_mask_erosion=cv2.erode(fg_mask,np.ones(ksize,np.uint8))

  #Detect contours
  contours_erosion,heirarchy=cv2.findContours(fg_mask_erosion,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)

  if len(contours_erosion) > 0:

    #Sort the contours (descending order)
    contour_sort=sorted(contours_erosion,key=cv2.contourArea,reverse=True)

    #Find the contour with largest area
    contour_area_max=cv2.contourArea(contour_sort[0])

    contour_frac=contour_area_max/frame_area

    if contour_frac > min_contour_area_thresh:

      #Setting the dimensions for the bounding box
      for idx in range(min(max_contours,len(contour_sort))):
        xc,yc,wc,hc=cv2.boundingRect(contour_sort[idx])
        if idx==0:   #contour with largest area
          x1=xc
          y1=yc
          x2=xc+wc
          y2=yc+hc
        else:
          x1=min(x1,xc)
          y1=min(y1,yc)
          x2=max(x2,xc+wc)
          y2=max(y2,yc+hc)

      #Draw the bounding box of the detected object
      cv2.rectangle(frame,(x1,y1),(x2,y2),yellow,thickness=2)

      #Post-processing (save the resultant video)
      vid_out.write(frame)

vid.release()
vid_out.release()

In [8]:
#Display of the Intrusion detection video (output video)
clip=VideoFileClip(vid_out_file)
clip.ipython_display(width=1000)

Moviepy - Building video __temp__.mp4.
Moviepy - Writing video __temp__.mp4



Moviepy - Done !
Moviepy - video ready __temp__.mp4
